# E3: LSTM and GRU on CIFAR-10 (replication notebook)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/broistg/deep-learning-CO5085/blob/main/E3/notebooks/E3.ipynb)

Thin notebook: implementation stays in `E3/e3lib/` and `common/`. Set `MODE` in the next cell:

| `MODE` | What it does | Runtime |
|---|---|---|
| `"load"` (default) | Runs the tests, regenerates tables and figures from `E3/results/`, and compares them with saved outputs when available | CPU |
| `"rerun"` | Smoke-tests a model, trains all 12 recurrent runs (four configurations × three seeds) into `E3/results_rerun/`, then regenerates analysis and compares with published results when available | GPU |

E3 uses `hidden_size=128` and `embed_dim=64`; `rerun` writes only to `E3/results_rerun/`. A rerun on another GPU need not match individual accuracies exactly. Protocol: `E3/README.md`.


In [ ]:
# 0. Settings
MODE = "load"  # "load" (CPU) or "rerun" (GPU, 12 full runs)
REPO_URL = "https://github.com/broistg/deep-learning-CO5085.git"
BRANCH = "main"
assert MODE in {"load", "rerun"}


In [ ]:
# 1. Locate or clone the repo, install dependencies on Colab, and expose E3 modules.
import os
import subprocess
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    repo = Path("/content/deep-learning-CO5085")
    if not (repo / ".git").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    repo = Path.cwd().resolve()
    while repo != repo.parent and not (repo / "common" / "cifar.py").exists():
        repo = repo.parent
    assert (repo / "common" / "cifar.py").exists(), "Run from inside the repository"

os.chdir(repo)
for module_path in (repo / "E3", repo):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))
os.environ["PYTHONPATH"] = os.pathsep.join((str(repo), str(repo / "E3")))
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/e.txt"], check=True)
from common import notebook_utils as nu
from e3lib.plan import all_cfgs
from e3lib.train import E3_DIR, run
from e3lib import analyze, diagnostics
print("Repository:", repo)


In [ ]:
# 2. Show mode, device and run count before doing any training.
import torch
print(f"E3: MODE={MODE} | torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Planned recurrent runs: {len(all_cfgs())}")
if MODE == "rerun":
    assert torch.cuda.is_available(), "Select a GPU runtime for the 12 full CIFAR-10 runs"
    h, n = nu.estimate_hours(E3_DIR / "results")
    if n:
        print(f"Published training estimate: {h:.1f} h for {n} runs on the original GPU, plus evaluation.")
    else:
        print("No published E3 timings yet; recurrent training may take several hours.")


In [ ]:
# 3. Tests: models, tokenizers, resume/analysis, and the shared CIFAR split.
for suite in ("E3/tests", "common/tests"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", suite, "-v"], check=True)


## Load mode


In [ ]:
# 4a. Regenerate tables and figures from saved E3 recurrent runs.
regenerated = E3_DIR / "regenerated"
published = E3_DIR / "results"
published_runs = []
generated_this_run = False
diagnostics_generated = False
if MODE == "load":
    published_runs = analyze.load_results(published)
    analyze.main(["--results", str(published), "--out", str(regenerated)])
    generated_this_run = True
    print(f"Loaded {len(published_runs)}/12 recurrent runs.")


In [ ]:
# 4b. Check regenerated outputs against the saved tables and summary, when published.
if MODE == "load" and published_runs:
    reference_files = [published / "summary.json", published / "tab_e3_summary.tex"]
    if all(path.is_file() for path in reference_files):
        nu.compare_outputs(regenerated, published)
    else:
        print("Saved E3 tables/summary are not available yet; regenerated outputs cannot be compared.")
    try:
        diagnostics.main(["--results", str(published), "--out", str(regenerated)])
        diagnostics_generated = True
    except ValueError as exc:
        print("Test-logit diagnostics unavailable:", exc)


## Rerun mode


In [ ]:
# 5a. Dry run on synthetic data; it writes no official result or checkpoint.
if MODE == "rerun":
    run(all_cfgs()[0], smoke=True)


In [ ]:
# 5b. Train all 12 recurrent runs into a separate directory. Re-running this cell resumes checkpoints and skips verified completed runs.
if MODE == "rerun":
    rerun_dir = E3_DIR / "results_rerun"
    for cfg in all_cfgs():
        run(cfg, out_dir=rerun_dir)


In [ ]:
# 5c. Compare with published accuracies when available; regenerate tables and diagnostics from the rerun.
if MODE == "rerun":
    if any(published.glob("*_s*.json")):
        analyze.load_results(published)
        nu.compare_means(rerun_dir, published)
        nu.compare_runs(rerun_dir, published)
    else:
        print("No published E3 recurrent results yet; rerun comparison skipped.")
    analyze.main(["--results", str(rerun_dir), "--out", str(regenerated)])
    generated_this_run = True
    diagnostics.main([
        "--results", str(rerun_dir),
        "--artifacts", str(rerun_dir / "artifacts"),
        "--out", str(regenerated),
    ])
    diagnostics_generated = True


## Figures and tables


In [ ]:
# 6. Show figures and tables generated in this notebook run.
import importlib.util
if generated_this_run:
    figures = [regenerated / "figures" / "curves_e3.png"]
    if diagnostics_generated:
        figures.append(regenerated / "figures" / "confusion_e3.png")
    if importlib.util.find_spec("IPython"):
        for figure in figures:
            nu.show_pngs(figure.parent, pattern=figure.name)
    else:
        for figure in figures:
            print("Figure:", figure)
    for path in sorted(regenerated.glob("tab_*.tex")):
        print("---", path.name)
        print(path.read_text(encoding="utf8"))
else:
    print("No E3 figures or tables have been regenerated yet.")


`E3/results/` and `E3/artifacts/` hold the hidden-128 recurrent runs. `rerun` keeps its results and artifacts under `E3/results_rerun/`; re-executing the notebook resumes them. Once saved tables and summary exist, `load` checks the regenerated numbers against those files. Test-logit diagnostics additionally need local `E3/artifacts/` files.
